[![img/pythonista.png](img/pythonista.png)](https://www.pythonista.io)

# Introducción a *Dask*.

* La siguiente celda instala *Dask* con soporte para *DataFrames* en el entorno actual en caso de que no esté disponible.

In [1]:
%pip install dask[dataframe]

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.


* La siguiente celda importa `polars` con el alias convencional `pl`, `dask.dataframe` con el alias `dd`, junto con `pandas`, `numpy` y `os`.

**Nota:** Por convención, `dask.dataframe` se importa con el alias `dd`.

In [31]:
import polars as pl
import dask.dataframe as dd
import numpy as np
import pandas as pd
import os

* La siguiente celda define y en su caso crea los directorios que se usarán para almacenar los datos y los resultados de este capítulo.

In [3]:
sample_dir = 'data'
temp_dir   = 'data_tmp'
os.makedirs(sample_dir, exist_ok=True)
os.makedirs(temp_dir,   exist_ok=True)

## Contexto: Del análisis local al escalado distribuido.

En los capítulos anteriores se estudiaron **Polars** y **PyArrow**, que ofrecen capacidades de alto rendimiento para analizar datos en una sola máquina. Sin embargo, cuando los datos crecen más allá de lo que puede caber en la memoria disponible, se necesita una estrategia diferente.

### ¿Polars o Dask?

| Aspecto | **Polars** | **Dask** |
|---------|-----------|----------|
| **Tamaño de datos** | Hasta ~100 GB en memoria | >100 GB (terabytes) |
| **Ejecución** | Una máquina | Múltiples máquinas (clúster) |
| **Velocidad (single-machine)** | 3-10× más rápido que *Pandas* | Overhead distribuido |
| **Facilidad de uso** | *API* simple y moderna | Requiere configuración de clúster |
| **Casos de uso** | ML local, analítica exploratoria | Big Data, ETL distribuido |
| **Integración Cloud** | Experimental | Madura (*AWS*, *GCP*, *Azure*) |

**Regla práctica:**
- Usar **Polars** cuando los datos caben en memoria (máxima velocidad).
- Usar **Dask** cuando los datos superan la capacidad de una máquina.

[*Dask*](https://dask.org/) es una biblioteca general para cómputo paralelo que permite escalar operaciones a través de clústers. A diferencia de *Polars*, que está optimizado para una sola máquina, *Dask* está diseñado para distribuir el trabajo entre múltiples equipos.

*Dask* consta de:

* Un calendarizador de tareas dinámico (*dynamic task scheduler*).
* Una colección de bibliotecas optimizadas para *Big Data* con interfaces que extienden a *NumPy* y *Pandas*.

https://docs.dask.org/en/stable/

https://tutorial.dask.org/

<img src="img/arquitectura_dask.png" width=75%>

## Principales paquetes de *Dask*.

### Colecciones de datos.

| Paquete | Alias | Descripción |
| :--- | :--- | :--- |
| [`dask.array`](https://docs.dask.org/en/stable/array.html) | `da` | Arreglos paralelos equivalentes a *NumPy*. |
| [`dask.dataframe`](https://docs.dask.org/en/stable/dataframe.html) | `dd` | *DataFrames* distribuidos equivalentes a *Pandas*. |
| [`dask.bag`](https://docs.dask.org/en/stable/bag.html) | `db` | Colecciones de datos semi-estructurados. |

### Bibliotecas de paralelismo.

| Paquete | Descripción |
| :--- | :--- |
| [`dask.delayed`](https://docs.dask.org/en/stable/delayed.html) | Paraleliza funciones de *Python* de forma declarativa. |
| [`dask.futures`](https://docs.dask.org/en/stable/futures.html) | Implementación de `concurrent.futures` optimizada para clústers. |

## Evaluación perezosa (*lazy*) con `df.compute()`.

Al igual que *Polars*, *Dask* utiliza **evaluación lazy**: las operaciones se definen sin ejecutarse hasta que se llama explícitamente a `.compute()`. La diferencia es que *Dask* distribuye esa ejecución entre los núcleos disponibles o entre los *workers* de un clúster.

| Operación | Comportamiento |
| :--- | :--- |
| [`dd.read_csv(path)`](https://docs.dask.org/en/stable/dataframe-api.html#dask.dataframe.read_csv) | Registra la intención de leer; **no carga datos**. |
| [`df.compute()`](https://docs.dask.org/en/stable/dataframe-api.html#dask.dataframe.DataFrame.compute) | Ejecuta el grafo y devuelve el resultado como *Pandas DataFrame*. |
| [`df.persist()`](https://docs.dask.org/en/stable/dataframe-api.html#dask.dataframe.DataFrame.persist) | Ejecuta el grafo y retiene el resultado en memoria (*caché*). |

## *Dataset* ilustrativo.

Para garantizar la reproducibilidad del notebook se genera un *dataset* sintético con series epidemiológicas: 200 registros diarios con columnas de casos por entidad (`Nacional`, `CDMX`, `Jalisco`, `NuevoLeon`).

* La siguiente celda usa **Polars** para crear el *dataset* de ejemplo y guardarlo como CSV. Este es el patrón recomendado: generar y preprocesar con *Polars* (rápido, en memoria), luego leer con *Dask* cuando se requiere escalar.

In [13]:
from datetime import date, timedelta

np.random.seed(42)
n = 200
start    = date(2020, 3, 1)
end      = start + timedelta(days=n - 1)
fechas   = pl.date_range(start, end, interval='1d', eager=True)
nacional = np.random.randint(5_000, 120_000, n)

df_covid = pl.DataFrame({
    'index':     list(range(n)),
    'Fecha':     fechas.dt.strftime('%Y-%m-%d'),
    'Nacional':  nacional.tolist(),
    'CDMX':      (nacional * np.random.uniform(0.20, 0.40, n)).astype(int).tolist(),
    'Jalisco':   (nacional * np.random.uniform(0.08, 0.15, n)).astype(int).tolist(),
    'NuevoLeon': (nacional * np.random.uniform(0.06, 0.12, n)).astype(int).tolist(),
})

url_covid_csv = os.path.join(sample_dir, 'covid.csv')
df_covid.write_csv(url_covid_csv)
print(f'Dataset: {df_covid.shape[0]} filas, {df_covid.shape[1]} columnas')
print(df_covid.head())

Dataset: 200 filas, 6 columnas
shape: (5, 6)
┌───────┬────────────┬──────────┬───────┬─────────┬───────────┐
│ index ┆ Fecha      ┆ Nacional ┆ CDMX  ┆ Jalisco ┆ NuevoLeon │
│ ---   ┆ ---        ┆ ---      ┆ ---   ┆ ---     ┆ ---       │
│ i64   ┆ str        ┆ i64      ┆ i64   ┆ i64     ┆ i64       │
╞═══════╪════════════╪══════════╪═══════╪═════════╪═══════════╡
│ 0     ┆ 2020-03-01 ┆ 20795    ┆ 4666  ┆ 1800    ┆ 2182      │
│ 1     ┆ 2020-03-02 ┆ 5860     ┆ 1589  ┆ 543     ┆ 489       │
│ 2     ┆ 2020-03-03 ┆ 108694   ┆ 41452 ┆ 15806   ┆ 11929     │
│ 3     ┆ 2020-03-04 ┆ 115268   ┆ 29327 ┆ 14371   ┆ 10851     │
│ 4     ┆ 2020-03-05 ┆ 81820    ┆ 26962 ┆ 9504    ┆ 5220      │
└───────┴────────────┴──────────┴───────┴─────────┴───────────┘


* La siguiente celda importa `dask.dataframe` e invoca `dd.read_csv()`. **Ningún dato se carga en este momento** — *Dask* registra la intención de leer el archivo y devuelve un *dataframe* lazy.

In [14]:
df_dask = dd.read_csv(url_covid_csv)
print(f'Tipo: {type(df_dask)}')
print(f'Particiones: {df_dask.npartitions}')

Tipo: <class 'dask.dataframe.dask_expr._collection.DataFrame'>
Particiones: 1


* La siguiente celda muestra el `DataFrame` de *Dask* (modo *lazy*): solo se presentan los metadatos (esquema y número de particiones), sin cargar ningún dato en memoria.

In [15]:
df_dask

,index,Fecha,Nacional,CDMX,Jalisco,NuevoLeon
npartitions=1,,,,,,
,int64,string,int64,int64,int64,int64
,...,...,...,...,...,...


* La siguiente celda llama a `.compute()` para materializar **todo** el CSV en un *DataFrame* de *Pandas*.

In [16]:
df_dask.compute()

,index,Fecha,Nacional,CDMX,Jalisco,NuevoLeon
0,0,2020-03-01,20795,4666,1800,2182
1,1,2020-03-02,5860,1589,543,489
2,2,2020-03-03,108694,41452,15806,11929
3,3,2020-03-04,115268,29327,14371,10851
4,4,2020-03-05,81820,26962,9504,5220
...,...,...,...,...,...,...
195,195,2020-09-12,53925,18210,5866,5988
196,196,2020-09-13,47941,10146,6444,5022
197,197,2020-09-14,26834,10278,3049,2267
198,198,2020-09-15,23047,6648,3433,2672


* La siguiente celda accede a la columna `'Nacional'` y verifica su tipo con `type()`. El resultado es una `dask.dataframe.Series` (aún *lazy*).

In [17]:
type(df_dask['Nacional'])

dask.dataframe.dask_expr._collection.Series

* La siguiente celda aplica `.compute()` solo a la columna `'Nacional'`, cargando únicamente esa serie en memoria en lugar de todo el *DataFrame*.

In [18]:
df_dask['Nacional'].compute()

0       20795
1        5860
2      108694
3      115268
4       81820
        ...  
195     53925
196     47941
197     26834
198     23047
199     31105
Name: Nacional, Length: 200, dtype: int64

* La siguiente celda encadena un filtro y una selección de columnas de forma *lazy*, sin ejecutar ningún cómputo. El grafo de tareas se construye pero no se evalúa.

In [19]:
query = df_dask.loc[df['Nacional'] > 50000].loc[:, ['index', 'Nacional']]
print(f'Tipo: {type(query)}')

Tipo: <class 'dask.dataframe.dask_expr._collection.DataFrame'>


* La siguiente celda ejecuta la cadena de operaciones *lazy* con `.compute()`, cargando en memoria únicamente las filas y columnas que cumplen el filtro.

In [20]:
query.compute()

,index,Nacional
2,2,108694
3,3,115268
4,4,81820
5,5,59886
7,7,87386
...,...,...
190,190,100003
191,191,100259
192,192,117235
193,193,74163


### Persistencia en memoria con `df.persist()`.

Con evaluación perezosa, cada llamada a `df.compute()` **recalcula todo el grafo** desde el origen. Cuando el mismo *dataframe* se reutiliza en múltiples operaciones, esto implica releer y reprocesar los datos innecesariamente.

`df.persist()` ejecuta el grafo **una sola vez** y retiene los resultados en memoria, comportándose como un caché distribuido.

| Operación | Comportamiento |
| :--- | :--- |
| `df.compute()` | Ejecuta el grafo y devuelve el resultado a *Python*. |
| `df.persist()` | Ejecuta el grafo y retiene el resultado en los *workers*. |

https://docs.dask.org/en/stable/api.html#dask.dataframe.DataFrame.persist

* La siguiente celda ilustra la diferencia entre `.compute()` y `.persist()`: sin `.persist()` cada operación relee el CSV desde disco; con `.persist()` los datos se cargan una sola vez y las operaciones siguientes operan sobre la copia en memoria.

* La siguiente celda no usa `persist()`, por lo que cada operación relee y recalcula desde disco.

In [21]:
from time import time
start_time = time()
print(df_dask[df_dask['Nacional'] > 50000].compute())
print(df_dask['Nacional'].mean().compute())
end_time = time()
print(f'Tiempo de ejecución: {end_time - start_time:.4f} segundos')

     index       Fecha  Nacional   CDMX  Jalisco  NuevoLeon
2        2  2020-03-03    108694  41452    15806      11929
3        3  2020-03-04    115268  29327    14371      10851
4        4  2020-03-05     81820  26962     9504       5220
5        5  2020-03-06     59886  11983     7545       3725
7        7  2020-03-08     87386  22803    11456       5314
..     ...         ...       ...    ...      ...        ...
190    190  2020-09-07    100003  34881    14434       6700
191    191  2020-09-08    100259  25082     8601       8861
192    192  2020-09-09    117235  27769    16573      11297
193    193  2020-09-10     74163  16032     8796       7984
195    195  2020-09-12     53925  18210     5866       5988

[127 rows x 6 columns]
63800.44
Tiempo de ejecución: 0.0177 segundos


* En la siguiente celda se define el *dataframe* persistente  `df_cache` con `persist()` y los datos se cargan una sola vez en memoria.   

In [22]:
df_cache = df_dask.persist()

* Las siguientes operaciones son más rápidas porque operan sobre la copia en memoria, sin necesidad de releer desde disco.

In [23]:
time_cache_start = time()
print(df_cache[df_cache['Nacional'] > 50000].compute())
print(df_cache['Nacional'].mean().compute())
time_cache_end = time()
print(f'Tiempo de ejecución con caché: {time_cache_end - time_cache_start:.4f} segundos')

     index       Fecha  Nacional   CDMX  Jalisco  NuevoLeon
2        2  2020-03-03    108694  41452    15806      11929
3        3  2020-03-04    115268  29327    14371      10851
4        4  2020-03-05     81820  26962     9504       5220
5        5  2020-03-06     59886  11983     7545       3725
7        7  2020-03-08     87386  22803    11456       5314
..     ...         ...       ...    ...      ...        ...
190    190  2020-09-07    100003  34881    14434       6700
191    191  2020-09-08    100259  25082     8601       8861
192    192  2020-09-09    117235  27769    16573      11297
193    193  2020-09-10     74163  16032     8796       7984
195    195  2020-09-12     53925  18210     5866       5988

[127 rows x 6 columns]
63800.44
Tiempo de ejecución con caché: 0.0077 segundos


### Otras bibliotecas de paralelismo.

* [`dask.delayed`](https://docs.dask.org/en/stable/delayed.html): permite paralelizar funciones arbitrarias de *Python* decorándolas con `@dask.delayed`. Útil para pipelines personalizados que no encajan en el modelo de *DataFrame*.
* [`dask.futures`](https://docs.dask.org/en/stable/futures.html): implementación de `concurrent.futures` optimizada para clústers *Dask*. Útil cuando se necesita control fino sobre la ejecución distribuida.

## Procesamiento de datos particionados.

El **particionado *Hive-style*** organiza archivos *Parquet* en subdirectorios según el valor de una columna:

```
datos/
  region=norte/
    part.0.parquet
  region=sur/
    part.0.parquet
  region=este/
    part.0.parquet
```

*Dask* puede leer únicamente las particiones necesarias (*partition pruning*), evitando leer archivos irrelevantes. Esto reduce significativamente el I/O en grandes volúmenes de datos.

https://docs.dask.org/en/stable/dataframe-parquet.html

* La siguiente celda convierte el *DataFrame* de *Polars* a una tabla de *PyArrow*, casteando las columnas `LargeUtf8` (tipo nativo de *Polars*) a `Utf8` estándar, y escribe el dataset particionado por `region` usando `pyarrow.parquet.write_to_dataset`. Este paso garantiza la compatibilidad de tipos con *Dask*.

In [36]:
import pyarrow as pa
import pyarrow.parquet as pq

ventas_url = 'data_tmp/ventas_particionadas'

df_ventas = pl.DataFrame({
    'region':   ['norte', 'norte', 'sur', 'sur', 'este'],
    'producto': ['A', 'B', 'A', 'C', 'B'],
    'ventas':   [100, 200, 150, 300, 250],
})

# Polars usa LargeUtf8; Dask/PyArrow espera Utf8.
# Se convierte la tabla a tipos estándar antes de particionar.
table = df_ventas.to_arrow()
schema_compatible = pa.schema([
    field.with_type(pa.string()) if pa.types.is_large_string(field.type) else field
    for field in table.schema
])
pq.write_to_dataset(
    table.cast(schema_compatible),
    root_path=ventas_url,
    partition_cols=['region'],
)
print('Datos escritos con particionado por región:')
print(df_ventas)

Datos escritos con particionado por región:
shape: (5, 3)
┌────────┬──────────┬────────┐
│ region ┆ producto ┆ ventas │
│ ---    ┆ ---      ┆ ---    │
│ str    ┆ str      ┆ i64    │
╞════════╪══════════╪════════╡
│ norte  ┆ A        ┆ 100    │
│ norte  ┆ B        ┆ 200    │
│ sur    ┆ A        ┆ 150    │
│ sur    ┆ C        ┆ 300    │
│ este   ┆ B        ┆ 250    │
└────────┴──────────┴────────┘


* La siguiente celda lee solo la partición `region='norte'` usando `filters=[('region', '==', 'norte')]`. *Dask* no abre los archivos de las demás regiones (*partition pruning*).

In [37]:
df_norte = dd.read_parquet(
    ventas_url,
    filters=[('region', '==', 'norte'),],
    engine='pyarrow'
)

print(f'Particiones leídas: {df_norte.npartitions}')
print(df_norte.compute())

Particiones leídas: 2
  producto  ventas region
0        A     100  norte
1        B     200  norte
0        A     100  norte
1        B     200  norte


## Despliegue de un clúster con `Dask.Distributed`.

*Dask* puede desplegarse en clústers mediante varios equipos *workers* gestionados por un *scheduler* central. Este modo distribuido es el que permite procesar volúmenes de datos a escala de *Big Data* y es la puerta de entrada a plataformas cloud como *AWS*, *GCP* y *Azure*.

https://distributed.dask.org/en/stable/

<img src="img/dask_cluster.png" width=45%>

Un clúster de *Dask* se compone de:

* **Scheduler**: coordina la ejecución del grafo de tareas.
* **Workers**: ejecutan las tareas individuales en paralelo.
* **Client**: interfaz de *Python* para enviar tareas al clúster.

Para uso local, `dask.distributed.LocalCluster` simula un clúster completo en la misma máquina. En producción, existen integraciones nativas con *Kubernetes*, *YARN*, *AWS Fargate*, *Google Cloud Dataproc* y *Azure ML*.

* La siguiente celda crea un clúster local con `LocalCluster` y un `Client` para conectarse a él. Esto simula un entorno distribuido completo en la misma máquina y es el punto de partida habitual antes de migrar a un clúster real en la nube.

In [38]:
from dask.distributed import Client, LocalCluster

cluster = LocalCluster()
client  = Client(cluster)
client

Connection method: Cluster object,Cluster type: distributed.LocalCluster
Dashboard: http://127.0.0.1:8787/status,
Dashboard: http://127.0.0.1:8787/status,Workers: 4
Total threads: 16,Total memory: 46.83 GiB
Status: running,Using processes: True
Comm: tcp://127.0.0.1:39631,Workers: 0
Dashboard: http://127.0.0.1:8787/status,Total threads: 0
Started: Just now,Total memory: 0 B
Comm: tcp://127.0.0.1:34625,Total threads: 4
Dashboard: http://127.0.0.1:43015/status,Memory: 11.71 GiB
Nanny: tcp://127.0.0.1:35665,


* La siguiente celda consulta la capacidad del clúster a través del `client`: número de *workers* activos, núcleos totales disponibles y memoria total y disponible por *worker*.

In [ ]:
info      = client.scheduler_info()
workers   = info['workers']
n_workers = len(workers)
n_cores   = sum(w['nthreads'] for w in workers.values())
mem_total = sum(w['memory_limit'] for w in workers.values())

print(f'Workers activos : {n_workers}')
print(f'Núcleos totales : {n_cores}')
print(f'Memoria total   : {mem_total / 1024**3:.2f} GiB')
print()
print(f'{"Worker":<45} {"Núcleos":>7} {"Memoria (GiB)":>14}')
print('-' * 68)
for addr, w in workers.items():
    mem_gib = w['memory_limit'] / 1024**3
    print(f'{addr:<45} {w["nthreads"]:>7} {mem_gib:>14.2f}')

### Demostración de actividad del clúster.

`client.dashboard_link` devuelve la URL del *dashboard* de *Dask*, que muestra en tiempo real:

| Panel | Descripción |
| :--- | :--- |
| **Task Stream** | Tareas ejecutadas por cada *worker* (color por tipo). |
| **Progress** | Barra de avance por colección de tareas. |
| **Workers** | Uso de CPU y memoria por nodo. |

* La siguiente celda imprime el enlace al *dashboard*, crea un arreglo distribuido de 4 000 × 4 000 elementos repartido en 64 *chunks* y ejecuta una normalización *z-score* seguida de una reducción global. El grafo de tareas resultante (~200 tareas) genera actividad visible en todos los paneles del *dashboard*.

In [ ]:
import dask.array as da

print(f'Dashboard: {client.dashboard_link}\n')
print('Abre el dashboard en el navegador antes de ejecutar .compute().\n')

# Arreglo 4 000 x 4 000 distribuido en 64 chunks (500 x 500)
x = da.random.random(size=(4_000, 4_000), chunks=(500, 500))

# Normalización z-score + reducción global (~200 tareas en el grafo)
resultado = ((x - x.mean(axis=0)) / (x.std(axis=0) + 1e-8)).mean().compute()

print(f'Media z-score: {resultado:.8f}')

In [ ]:
client.close()
cluster.close()

## Resumen.

*Dask* complementa a *Polars* como capa de escalado distribuido, utilizando el mismo paradigma de evaluación lazy y el formato *Parquet* como estándar de interoperabilidad:

| Concepto | Descripción |
| :--- | :--- |
| **`dd.read_csv()` / `dd.read_parquet()`** | Lectura *lazy*: registra la intención sin cargar datos. |
| **`.compute()`** | Materializa el grafo y devuelve un *DataFrame* de *Pandas*. |
| **`.persist()`** | Cachea el resultado en memoria para reutilización eficiente. |
| **Particionado *Hive-style*** | Organización de *Parquet* por columna para *partition pruning*. |
| **`dask.distributed`** | Clúster distribuido; puerta de entrada a *Cloud* y *Big Data*. |
| **Flujo recomendado** | Polars (local) → Parquet → Dask (distribuido). |

<p style="text-align: center"><a rel="license" href="http://creativecommons.org/licenses/by/4.0/"><img alt="Licencia Creative Commons" style="border-width:0" src="https://i.creativecommons.org/l/by/4.0/80x15.png" /></a><br />Esta obra está bajo una <a rel="license" href="http://creativecommons.org/licenses/by/4.0/">Licencia Creative Commons Atribución 4.0 Internacional</a>.</p>
<p style="text-align: center">&copy; José Luis Chiquete Valdivieso. 2017-2026.</p>